In [6]:
# Install required library (uncomment if needed)
# !pip install opensearch-py

from opensearchpy import OpenSearch
import json
import warnings

# Suppress SSL warnings
warnings.filterwarnings('ignore')

config = {
    'hosts': ['https://os-ms-dev.cwsystem.in/'],
    'http_auth': ('cw', 'Carwale@123'),
    'use_ssl': True,
    'verify_certs': False,
    'ssl_assert_hostname': False,
    'ssl_show_warn': False,
    'timeout': 3000
}

# Create OpenSearch client
client = OpenSearch(**config)

# Test connection
try:
    info = client.info()
    print("Connected to OpenSearch successfully!")
    print(f"Cluster: {info['cluster_name']}")
    print(f"Version: {info['version']['number']}")
except Exception as e:
    print(f"Failed to connect: {e}")

Connected to OpenSearch successfully!
Cluster: 408531640850:opensearch-ms-dev
Version: 7.10.2


In [7]:
def upload_to_opensearch_index(client, source_file, index_name, batch_size=500):
    """
    Upload documents to OpenSearch index from final_data.json.
    Works with flat array format where documents are not wrapped in _source.
    
    Args:
        client: OpenSearch client instance
        source_file: Path to the JSON file containing documents
        index_name: Name of the target index in OpenSearch
        batch_size: Number of documents per bulk upload batch
    
    Returns:
        dict: Statistics about the upload operation
    """
    from opensearchpy import helpers
    
    print(f"{'='*60}")
    print(f"Starting upload to index: {index_name}")
    print(f"{'='*60}\n")
    
    # Load the source data
    print("Loading source data...")
    with open(source_file, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    print(f"Loaded {len(data)} documents from {source_file}\n")
    
    # Check if index exists
    if client.indices.exists(index=index_name):
        print(f"Index '{index_name}' already exists.")
        response = input("Do you want to delete and recreate it? (yes/no): ")
        if response.lower() in ['yes', 'y']:
            client.indices.delete(index=index_name)
            print(f"Deleted existing index '{index_name}'")
        else:
            print("Aborting upload. Index already exists.")
            return None
    
    # Create the new index with settings to handle large nested objects
    print(f"Creating index '{index_name}'...")
    index_settings = {
        "settings": {
            "index.mapping.total_fields.limit": 5000,
            "index.mapping.nested_fields.limit": 100,
            "index.mapping.depth.limit": 20
        }
    }
    client.indices.create(index=index_name, body=index_settings)
    print(f"Index '{index_name}' created successfully with increased field limits\n")
    
    # Prepare documents for bulk upload
    print("Preparing documents for upload...")
    actions = []
    
    for record in data:
        # Use the record directly as it's already in the correct format
        doc_content = record
        
        # Use vehicle_id as the document ID for uniqueness
        doc_id = doc_content.get('vehicle_id', '')
        
        if doc_id:
            action = {
                '_index': index_name,
                '_id': doc_id,
                '_source': doc_content
            }
            actions.append(action)
    
    print(f"Prepared {len(actions)} documents for upload\n")
    
    # Bulk upload with progress tracking and error capture
    print("Starting bulk upload...")
    success_count = 0
    error_count = 0
    errors = []
    failed_docs = []
    
    # Process in batches
    for i in range(0, len(actions), batch_size):
        batch = actions[i:i+batch_size]
        batch_errors = 0
        batch_success = 0
        
        try:
            for ok, response in helpers.streaming_bulk(client, batch, raise_on_error=False):
                if ok:
                    batch_success += 1
                    success_count += 1
                else:
                    batch_errors += 1
                    error_count += 1
                    errors.append(response)
                    if 'index' in response and '_id' in response['index']:
                        failed_docs.append(response['index']['_id'])
            
            print(f"Progress: {min(i+batch_size, len(actions))}/{len(actions)} documents processed "
                  f"(Batch: +{batch_success} success, +{batch_errors} errors | "
                  f"Total: {success_count} success, {error_count} errors)")
        except Exception as e:
            print(f"Critical error in batch {i}-{min(i+batch_size, len(actions))}: {e}")
            error_count += len(batch)
            batch_errors = len(batch)
    
    # Save errors to file if any occurred
    if errors:
        error_file = 'upload_errors.json'
        print(f"\nSaving {len(errors)} errors to {error_file}...")
        with open(error_file, 'w', encoding='utf-8') as f:
            json.dump(errors, f, indent=2)
        print(f"Errors saved to {error_file}")
        
        # Analyze error types
        error_types = {}
        for error in errors:
            if 'index' in error and 'error' in error['index']:
                error_info = error['index']['error']
                if isinstance(error_info, dict):
                    error_type = error_info.get('type', 'unknown')
                    error_reason = error_info.get('reason', 'no reason provided')
                else:
                    error_type = 'unknown'
                    error_reason = str(error_info)
                
                key = f"{error_type}: {error_reason[:100]}"
                error_types[key] = error_types.get(key, 0) + 1
        
        print(f"\n{'='*60}")
        print("Error Type Breakdown:")
        print(f"{'='*60}")
        for error_type, count in sorted(error_types.items(), key=lambda x: x[1], reverse=True)[:5]:
            print(f"{count:4d} - {error_type}")
    
    # Retry failed documents if any
    if failed_docs and error_count > 0:
        print(f"\n{'='*60}")
        print(f"Retrying {len(failed_docs)} failed documents...")
        print(f"{'='*60}\n")
        
        retry_actions = [action for action in actions if action['_id'] in failed_docs]
        retry_success = 0
        retry_errors = []
        
        for action in retry_actions:
            try:
                for ok, response in helpers.streaming_bulk(client, [action], raise_on_error=False):
                    if ok:
                        retry_success += 1
                    else:
                        retry_errors.append(response)
            except Exception as e:
                retry_errors.append({'error': str(e), '_id': action['_id']})
        
        success_count += retry_success
        error_count -= retry_success
        
        print(f"Retry complete: {retry_success} recovered, {len(retry_errors)} still failed")
        
        if retry_errors:
            retry_error_file = 'upload_retry_errors.json'
            with open(retry_error_file, 'w', encoding='utf-8') as f:
                json.dump(retry_errors, f, indent=2)
            print(f"Persistent errors saved to {retry_error_file}")
    
    # Refresh the index to make documents searchable
    client.indices.refresh(index=index_name)
    
    # Get final count from index
    count_response = client.count(index=index_name)
    final_count = count_response['count']
    
    stats = {
        'total_documents': len(data),
        'documents_prepared': len(actions),
        'success_count': success_count,
        'error_count': error_count,
        'final_index_count': final_count,
        'index_name': index_name,
        'errors': errors if errors else []
    }
    
    print(f"\n{'='*60}")
    print(f"Upload Complete!")
    print(f"{'='*60}")
    print(f"Total documents in source file: {stats['total_documents']}")
    print(f"Documents prepared for upload: {stats['documents_prepared']}")
    print(f"Successfully uploaded: {stats['success_count']}")
    print(f"Errors: {stats['error_count']}")
    print(f"Final document count in index: {stats['final_index_count']}")
    print(f"Success rate: {(stats['success_count']/stats['total_documents']*100):.1f}%")
    
    return stats

In [8]:
# Upload documents to the new index
upload_stats = upload_to_opensearch_index(
    client=client,
    source_file='final_data.json',
    index_name='mcp_version_data_v4',
    batch_size=500
)

Starting upload to index: mcp_version_data_v4

Loading source data...
Loaded 2017 documents from final_data.json

Index 'mcp_version_data_v4' already exists.
Deleted existing index 'mcp_version_data_v4'
Creating index 'mcp_version_data_v4'...
Index 'mcp_version_data_v4' created successfully with increased field limits

Preparing documents for upload...
Prepared 2017 documents for upload

Starting bulk upload...
Progress: 500/2017 documents processed (Batch: +500 success, +0 errors | Total: 500 success, 0 errors)
Progress: 1000/2017 documents processed (Batch: +500 success, +0 errors | Total: 1000 success, 0 errors)
Progress: 1500/2017 documents processed (Batch: +500 success, +0 errors | Total: 1500 success, 0 errors)
Progress: 2000/2017 documents processed (Batch: +500 success, +0 errors | Total: 2000 success, 0 errors)
Progress: 2017/2017 documents processed (Batch: +17 success, +0 errors | Total: 2017 success, 0 errors)

Upload Complete!
Total documents in source file: 2017
Document